## 2. Preprocessing
This section provides a continuation of the initial examination of the dataset compiled by Mesquita and Pires (2024)[1].

This notebook explores the next 20 most cited resolutions. We cross the Mesquita and Pires data [1] with a database of scrapped resolutions on the online UN resource: documents.un.org.

Moreover, we sort the databse with respect of number of received citations and export it for another analysis of topic mining using the official united nations document online library.

[1] Rafael Mesquita, Antonio Pires, The references of the nations: Introducing a corpus of United Nations General Assembly resolutions since 1946 and their citation network, Journal of Peace Research, Volume 62, Issue 4, July 2025, Pages 1279–1291, https://doi.org/10.1177/00223433241254997

In [ ]:
import os
import pandas as pd
import re

# ----------------------------
# SETTINGS
# ----------------------------
CITATIONS_CSV = "./ga_citations_1946_2019.csv"
RESOLUTIONS_CSV = "./ga_resolutions_1946_2019-subjects.csv"

# ----------------------------
# LOAD DATA
# ----------------------------
if not os.path.isfile(CITATIONS_CSV):
    print(f"Error: File '{CITATIONS_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 200000
        chunks = []

        for chunk in pd.read_csv(CITATIONS_CSV, chunksize=chunksize):
            chunks.append(chunk)

        df1 = pd.concat(chunks, ignore_index=True)
        print("Citations file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

    finally:
        pass

if not os.path.isfile(RESOLUTIONS_CSV):
    print(f"Error: File '{RESOLUTIONS_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 200000
        chunks = []

        for chunk in pd.read_csv(RESOLUTIONS_CSV, chunksize=chunksize):
            chunks.append(chunk)

        df2 = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

    finally:
        pass

In [ ]:

print('Columns in df1. The most detailed dataframe, with citated and citators:')
print(df1.columns)


print('Columns in df2. A more general dataframe, with the official titles of resolution from the documents.un.org:')
print(df2.columns)

In [ ]:
# Important IDs: 'res_id2_doc_receiv_cite', 'res_id3_recv'

print("Let's take a look of the top 30 to 50 most cited resolutions.\n")
top_cited = (
    df1.groupby(['topic_recv', 'res_id2_doc_receiv_cite', 'res_id2_unlet_recv', 'res_id3_recv'])
      .size()
      .reset_index(name='citation_count')
      .sort_values(by='citation_count', ascending=False)
)

# Get top 30
top_50 = top_cited.head(50)

top_30_50 = top_50.tail(20)

print('\n\n',top_30_50[['topic_recv', 'citation_count', 'res_id3_recv']])

In [ ]:
pd.set_option('display.max_colwidth', None)
#order by resid
top_30_50 = top_30_50.sort_values(by='res_id2_unlet_recv')
print('Titles under mesquita and pires database:\n', top_30_50['topic_recv'],'\n\n')

titles = df2[df2['res_id'].isin(top_30_50['res_id2_unlet_recv'])]
titles = titles.sort_values(by='res_id')
print('Titles under documents.un.org:\n', titles['title'])
pd.reset_option('display.max_colwidth')


The titles in the documents.un.org resource are quite similar to the ones that appear in the field 'topic' of the Mesquita and Pires [1] database. There are slighly different versions, like second and 2nd.

It is worthy to check if inside the content both versions appear.

In [ ]:
print('Titles with resolution:\n', top_30_50[['topic_recv', 'res_id2_unlet_recv']])

In [ ]:
title = 'principles which should guide members in determining whether or not an obligation exists to transmit the information called for under article'
res_id = '1541 (xv)'
res_date = pd.to_datetime(
    df1.loc[df1['res_id2_unlet_recv'] == res_id, 'date_p_recv']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

In [ ]:
title = 'privileges and immunities of the united nations'
res_id = '22 (i)'
res_date = pd.to_datetime(
    df1.loc[df1['res_id2_unlet_recv'] == res_id, 'date_p_recv']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())